# Part A: Conditional DCGAN on CIFAR10

One sentence stating the goal: build a class-conditional DCGAN that generates 1000 CIFAR10 images (100 per class), as a second generative approach alongside the CVAE for comparison.

## 1. Imports and Setup

In [ ]:
# Import tensorflow/keras, numpy, matplotlib; set random seed for reproducibility.

## 2. Approach

One sentence describing why a class-conditional DCGAN is a suitable second approach here, noting unconditional GANs on all 10 CIFAR10 classes at once are notably harder to converge than single-class, which is why conditioning is treated as part of the baseline rather than an experiment.

## 3. Preprocessing

### 3.1 Normalize Pixel Values

One sentence noting we're scaling pixels to [-1, 1] rather than [0, 1] here, since the generator's output layer uses tanh (not sigmoid like the VAE decoder).

In [ ]:
# Cast images to float32 and normalize pixel values to [-1, 1] to match the generator's tanh output range.

### 3.2 Encode Labels

One sentence noting we're one-hot encoding the 10 class labels so they can be concatenated into the generator and discriminator as the conditioning signal, same approach as the CVAE.

In [ ]:
# One-hot encode the 10 class labels for use as the conditioning input.

### 3.3 Train/Validation Split

One sentence noting we're carving a validation set out of the 50k training images, used for the discriminator-accuracy diagnostic and to keep this preprocessing identical to the CVAE baseline's.

In [ ]:
# Shuffle the training set and split off a validation set (e.g. 45k/5k).

## 4. Build the Conditional Generator

### 4.1 What It Does

One sentence explaining that a noise vector and the one-hot label are concatenated, dense-expanded, and upsampled via transposed convs into a 32x32x3 image — followed by a diagram of this layer flow.

In [ ]:
# Diagram: generator layer flow — noise z + label -> concat -> dense expand -> reshape -> transposed conv stack (BatchNorm + ReLU) -> tanh output.

### 4.2 Implementation

No optimization tricks here (no spectral norm, no custom learning rate) — this is a plain conditional generator; tuning is reserved for `gan_improvement.ipynb`.

In [ ]:
# Build the generator: concatenate noise vector with label, dense-expand, then transposed convs (strided, BatchNorm, ReLU) back to 32x32x3, tanh output.

## 5. Build the Conditional Discriminator

### 5.1 What It Does

One sentence explaining that the image and the one-hot label (broadcast to a spatial map) are concatenated as input, then strided convs reduce it to a single real/fake logit — followed by a diagram of this layer flow.

In [ ]:
# Diagram: discriminator layer flow — image + label map -> concat -> strided conv stack (LeakyReLU) -> flatten -> dense -> real/fake logit.

### 5.2 Implementation

Plain conditional discriminator — no spectral normalization, no label smoothing baked in here; those are experiments, not baseline defaults.

In [ ]:
# Build the discriminator: concatenate image with a broadcast label map, strided conv stack (LeakyReLU, dropout), flatten, dense to a single logit.

## 6. Define the GAN Training Step

### 6.1 Loss Formulation

One sentence explaining the discriminator is trained to separate real from generated images (binary cross-entropy, hard 0/1 targets — no label smoothing in the baseline) while the generator is trained to fool it, and why both losses are tracked separately rather than combined into one number.

### 6.2 Architecture Pseudocode

```
train_step(images, labels):
    noise = random_normal(batch_size, latent_dim)

    with gradient_tape() as disc_tape:
        fake_images = generator(noise, labels)
        real_logits = discriminator(images, labels)
        fake_logits = discriminator(fake_images, labels)
        disc_loss = bce(ones_like(real_logits), real_logits) + bce(zeros_like(fake_logits), fake_logits)
    disc_gradients = disc_tape.gradient(disc_loss, discriminator.weights)
    disc_optimizer.apply_gradients(disc_gradients)

    noise = random_normal(batch_size, latent_dim)
    with gradient_tape() as gen_tape:
        fake_images = generator(noise, labels)
        fake_logits = discriminator(fake_images, labels)
        gen_loss = bce(ones_like(fake_logits), fake_logits)
    gen_gradients = gen_tape.gradient(gen_loss, generator.weights)
    gen_optimizer.apply_gradients(gen_gradients)

    return gen_loss, disc_loss
```

### 6.3 Implementation

Plain BCE loss, hard targets, one Adam optimizer per network at the same default learning rate — no TTUR, no label smoothing (those are Experiments 1 and 2 in `gan_improvement.ipynb`).

In [ ]:
# Define the GAN keras.Model subclass with a custom train_step implementing the alternating generator/discriminator updates above.

## 7. Train the Model

### 7.1 Fit and Save Best Weights

One sentence noting GANs have no single validation loss to checkpoint against (unlike the VAE), so weights are saved at a fixed final epoch after confirming training didn't visibly collapse in Section 8.

In [ ]:
# Compile and fit the GAN for a fixed number of epochs, saving the generator's weights to a .h5 checkpoint at the end.

### 7.2 Loss Curves

One sentence noting generator and discriminator loss are plotted on the same axes (not separately), since it's the balance between them — not either curve's absolute value — that indicates healthy GAN training.

In [ ]:
# Plot generator loss and discriminator loss on the same axes across training epochs.

## 8. Diagnostics

### 8.1 Fixed-Noise Progression Grid

One sentence noting the same fixed noise vector + label set is decoded every N epochs during training, producing a grid that shows the model visibly learning over time rather than just a before/after snapshot.

In [ ]:
# Generate images from a fixed noise+label batch every N epochs during training; display the saved snapshots as a progression grid.

### 8.2 Mode Collapse Quantification

One sentence noting we're computing pairwise pixel-distance (or std-dev) across a batch of same-class generated images as a numeric diversity check, rather than relying on eyeballing a sample grid for repeated images.

In [ ]:
# Generate a batch of images for one class, compute average pairwise pixel distance (or per-pixel std-dev) as a diversity score.

### 8.3 Nearest-Neighbor Check

One sentence noting we're comparing a handful of generated images against their closest real training image (by pixel distance) to check the generator is synthesizing, not memorizing, the training data.

In [ ]:
# For a few generated images, find and display the nearest real training image by pixel distance, side by side.

## 9. Generate 1000 Class-Conditioned Images

### 9.1 Sample and Generate

One sentence noting generation samples fresh noise vectors from the prior (not real images) and pairs them with a chosen class label, run through the generator only.

In [ ]:
# Sample random noise vectors, generate with each of the 10 class labels to produce 100 images per class (1000 total).

### 9.2 Save to Disk

One sentence noting all 1000 images are written to disk organized by class folder, as required for submission.

In [ ]:
# Save the generated images to disk as required for submission.

### 9.3 Preview Grid

One sentence noting we're displaying a small sample (a few per class) inline for a visual sanity check, separate from the full 1000 saved to disk.

In [ ]:
# Display a grid of sample generated images, a few per class, for visual inspection.

## 10. Evaluate Generated Image Quality

### 10.1 Eye-Test Scoring

One sentence noting we're manually scoring a sample of the saved generated images (e.g. 10 per class) as clear/marginal/nonsense, using the identical scoring criteria as the CVAE baseline so the two architectures are directly comparable later; FID is added in Section 10.5 as a quantitative complement.

In [ ]:
# Manually score a sample of generated images per class as clear / marginal / nonsense, tally results.

### 10.2 Per-Class Score Summary

One sentence noting we're tallying the eye-test scores by class into a bar chart, giving the evidence base for the discussion below.

In [ ]:
# Tally clear/marginal/nonsense counts per class, bar chart the result.

### 10.3 Discussion: Class Difficulty

One sentence noting which classes scored best/worst on the eye-test here, and whether the pattern matches or differs from the CVAE's class-difficulty findings.

### 10.4 Discussion: Color vs. Black-and-White (Prediction)

One sentence giving a reasoned prediction only (no grayscale model trained here) on whether black-and-white generation would be easier or harder for this GAN — this prediction is tested empirically in `gan_improvement.ipynb` Section 7 (Experiment 4: Color vs. Grayscale), which is the notebook to cite for the actual answer.

## 11. Baseline Conclusion

One sentence summarizing baseline GAN performance and key takeaways, plus saving the baseline weights/metrics to disk so `gan_improvement.ipynb` can load them without retraining.

In [ ]:
# Save baseline config, final losses, eye-test scores, and FID score to a small JSON file for gan_improvement.ipynb to load.

In [ ]:
# Extract InceptionV3 features for a sample of real validation images and for the 1000 generated images.

In [ ]:
# Fit a Gaussian (mean, covariance) to each feature set, compute Frechet distance between them, print the FID score.

## 11. Baseline Conclusion

One sentence summarizing baseline GAN performance and key takeaways, plus saving the baseline weights/metrics to disk so `gan_improvement.ipynb` can load them without retraining.

In [ ]:
# Save baseline config, final losses, and eye-test scores to a small JSON file for gan_improvement.ipynb to load.